# 依赖注入

学习目标：复用接口所需的参数和检查，理解依赖的调用次数，并在请求成功或失败时正确释放资源。

前置知识：Python 函数、类型标注、FastAPI 路由、请求校验、yield、上下文管理器与异常处理。

运行环境：Python 3.12、FastAPI 0.141.1；末节的 scope 参数要求 FastAPI 0.121.0 及以上。

环境准备：见 [FastAPI 环境与运行说明](README.md)。

工作目录：content/Web与应用开发/FastAPI。选择课程环境的 Python 3 (ipykernel)，从空内核顺序运行；TestClient 在进程内调用应用，每次使用 with 关闭客户端。临时文件由依赖关闭并自动删除。

## 1 把一组请求参数交给 Depends

多个列表接口可能都需要分页参数。先把 offset（跳过多少条）和 limit（最多取多少条）放进 page_params，再让接口声明自己需要它的返回值。这种由框架准备所需对象、再传给函数的方式称为依赖注入（Dependency Injection）。

Annotated 中的 dict[str, int] 描述收到的值，Depends(page_params) 指定负责提供它的函数。这里传函数本身；FastAPI 收到请求后解析其参数、执行函数，再把返回值传给路由。

Query 仍负责查询参数的约束。示例约定 offset 至少为 0，limit 在 1～20 之间。

In [1]:
from typing import Annotated

from fastapi import Depends, FastAPI, Query
from fastapi.testclient import TestClient

app = FastAPI()


def page_params(
    offset: Annotated[int, Query(ge=0)] = 0,
    limit: Annotated[int, Query(ge=1, le=20)] = 5,
) -> dict[str, int]:
    return {"offset": offset, "limit": limit}


@app.get("/records")
def list_records(page: Annotated[dict[str, int], Depends(page_params)]):
    return page


with TestClient(app) as client:
    response = client.get("/records", params={"offset": 2, "limit": 3})
# page 收到依赖返回的字典；这两个参数仍来自请求的查询部分。
assert response.json() == {"offset": 2, "limit": 3}
print(response.status_code, response.json())  # 预期：200 {'offset': 2, 'limit': 3}。

200 {'offset': 2, 'limit': 3}


C:\Users\ZHUANG\miniconda3\envs\hands-on-computing\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


继续使用本章的 app 和 page_params，为第二条路由声明同一依赖。参数默认值与校验规则都由这一处定义提供。

In [2]:
@app.get("/tags")
def list_tags(page: Annotated[dict[str, int], Depends(page_params)]):
    return page


with TestClient(app) as client:
    default_page = client.get("/tags")
    invalid_page = client.get("/tags", params={"limit": 0})
assert default_page.json() == {"offset": 0, "limit": 5}
assert invalid_page.status_code == 422
# 依赖中的校验也会阻止非法请求；loc 指明错误在查询参数 limit。
print(default_page.json())  # 预期：{'offset': 0, 'limit': 5}。
print(invalid_page.status_code, invalid_page.json()["detail"][0]["loc"])  # 预期：422 ['query', 'limit']。

{'offset': 0, 'limit': 5}
422 ['query', 'limit']


## 2 依赖也可以依赖别的函数

依赖函数的参数也能使用 Depends，这就形成了子依赖。下面的 page_description 需要先取得 page_params 的结果；FastAPI 按这个关系先准备分页参数，再生成描述，最后执行路由。

依赖关系由函数参数声明，路由只接收自己需要的描述文字。

In [3]:
def page_description(
    page: Annotated[dict[str, int], Depends(page_params)],
) -> str:
    return f"跳过 {page['offset']} 条，最多读取 {page['limit']} 条"


@app.get("/page-description")
def read_description(text: Annotated[str, Depends(page_description)]):
    return {"description": text}


with TestClient(app) as client:
    response = client.get("/page-description", params={"limit": 2})
# 请求参数经过 page_params → page_description → read_description。
assert response.json() == {"description": "跳过 0 条，最多读取 2 条"}
print(response.json())  # 预期：{'description': '跳过 0 条，最多读取 2 条'}。

{'description': '跳过 0 条，最多读取 2 条'}


## 3 请求内复用与 use_cache

同一请求在多个位置需要同一个依赖时，FastAPI 默认保存首次得到的值并复用，包括通过子依赖使用的情况。因此，两处 Depends 不一定对应两次调用，缓存也不会跨请求共享。

![同一次请求：共享依赖只求值一次](image/illustration/05-01-dependency-cache.svg)

图示：同一请求的依赖结果复用图。箭头表示值传递，不表示依赖之间需要网络调用。

下面的 calls 只记录本实验的调用次数，每次请求顺序完成。先核对两参数都为 1 且 calls 只有一项，再对照后面 use\_cache=False 的行为。

In [4]:
calls = []


def next_call_number() -> int:
    calls.append("called")
    return len(calls)


@app.get("/cached")
def read_cached(
    first: Annotated[int, Depends(next_call_number)],
    second: Annotated[int, Depends(next_call_number)],
):
    return {"first": first, "second": second}


calls.clear()
with TestClient(app) as client:
    response = client.get("/cached")
# 两个参数使用同一个依赖，本次请求只调用一次。
assert len(calls) == 1
assert response.json() == {"first": 1, "second": 1}
print(response.json(), "调用次数：", len(calls))  # 预期：{'first': 1, 'second': 1} 调用次数： 1。

{'first': 1, 'second': 1} 调用次数： 1


再发一次请求，保留 calls 的记录，检查依赖会不会重新执行。应用对象仍然是同一个。

In [5]:
with TestClient(app) as client:
    response = client.get("/cached")
# 新请求再次调用依赖：请求内缓存不会把上次请求的 1 沿用过来。
assert len(calls) == 2
assert response.json() == {"first": 2, "second": 2}
print(response.json(), "累计调用次数：", len(calls))  # 预期：{'first': 2, 'second': 2} 累计调用次数： 2。

{'first': 2, 'second': 2} 累计调用次数： 2


某个使用位置确实需要重新执行时，在该处设置 use_cache=False。它让这个位置重新调用依赖；其他位置仍按各自的声明处理。

In [6]:
@app.get("/uncached")
def read_uncached(
    first: Annotated[int, Depends(next_call_number)],
    second: Annotated[int, Depends(next_call_number, use_cache=False)],
):
    return {"first": first, "second": second}


calls.clear()
with TestClient(app) as client:
    response = client.get("/uncached")
# 第二个位置要求重新调用，因而单次请求累计调用两次。
assert len(calls) == 2
assert response.json() == {"first": 1, "second": 2}
print(response.json(), "调用次数：", len(calls))  # 预期：{'first': 1, 'second': 2} 调用次数： 2。

{'first': 1, 'second': 2} 调用次数： 2


## 4 只执行检查：路由与应用级依赖

路由需要一个检查，但不使用它的返回值时，可把 Depends 放入装饰器的 dependencies 列表。依赖仍会解析参数、调用子依赖，并可通过异常终止请求。

下面约定实验请求携带 X-Lab: notebook。它只是本实验的请求标记，不具备身份认证作用；Header 把 x_lab 对应到 X-Lab 请求头。

In [7]:
from fastapi import Header, HTTPException


def require_lab(x_lab: Annotated[str | None, Header()] = None) -> None:
    if x_lab != "notebook":
        raise HTTPException(status_code=400, detail="需要 X-Lab: notebook")


@app.get("/checked", dependencies=[Depends(require_lab)])
def read_checked():
    return {"message": "检查通过"}


with TestClient(app) as client:
    missing = client.get("/checked")
    accepted = client.get("/checked", headers={"X-Lab": "notebook"})
# 路由函数没有声明 x_lab，检查仍会执行；缺失标记时返回自定义错误。
assert missing.status_code == 400
assert accepted.json() == {"message": "检查通过"}
print(missing.status_code, missing.json())  # 预期：400 {'detail': '需要 X-Lab: notebook'}。
print(accepted.status_code, accepted.json())  # 预期：200 {'message': '检查通过'}。

400 {'detail': '需要 X-Lab: notebook'}
200 {'message': '检查通过'}


把 dependencies 配置在 FastAPI 实例上，会将它应用到该应用声明的所有路径操作。下面另建一个只有两条路由的 checked_app，继续复用 require_lab。

“应用级”描述的是检查覆盖哪些路由；这些依赖仍在处理请求时执行，不能把它理解为应用启动时只执行一次。

In [8]:
checked_app = FastAPI(dependencies=[Depends(require_lab)])


@checked_app.get("/records")
def checked_records():
    return {"items": []}


@checked_app.get("/tags")
def checked_tags():
    return {"tags": []}


with TestClient(checked_app) as client:
    for path in ("/records", "/tags"):
        missing = client.get(path)
        accepted = client.get(path, headers={"X-Lab": "notebook"})
        assert missing.status_code == 400
        assert accepted.status_code == 200
        # 两条路由都继承了应用级检查；每个新请求仍须提供标记。
        print(path, missing.status_code, accepted.status_code)  # 预期：/records 与 /tags 均先返回 400，再返回 200。

/records 400 200
/tags 400 200


## 5 用 yield 提供并关闭资源

参数可以直接 return；文件等资源还需要退出时清理。依赖先创建资源，用一次 yield 把资源交给使用者，最后在 finally 中关闭。FastAPI 会按上下文管理器的方式管理这个生成器，不需要为依赖函数添加 contextmanager 装饰器。

TemporaryFile 创建临时文件，关闭后自动删除；这里使用 UTF-8 文本模式。file_events 记录打开和关闭时实际读取到的 closed 属性，Iterator[TextIO] 表示生成器提供文本文件对象。

In [9]:
from collections.abc import Iterator
from tempfile import TemporaryFile
from typing import TextIO

file_events = []


def open_note() -> Iterator[TextIO]:
    stream = TemporaryFile(mode="w+", encoding="utf-8")
    try:
        file_events.append(("open", stream.closed))
        stream.write("今天练习依赖注入")
        stream.seek(0)
        yield stream
    finally:
        stream.close()
        file_events.append(("close", stream.closed))

在本章 app 上添加一条读取临时文本的路由。注入的 note 是 yield 提供的文件对象；路由只负责读取内容，关闭工作归 open_note 管理。

In [10]:
@app.get("/note")
def read_note(note: Annotated[TextIO, Depends(open_note)]):
    return {"text": note.read()}


file_events.clear()
with TestClient(app) as client:
    response = client.get("/note")
# 客户端调用完成后检查：打开时未关闭，退出时已关闭。
assert response.json() == {"text": "今天练习依赖注入"}
assert file_events == [("open", False), ("close", True)]
print(response.json())  # 预期：{'text': '今天练习依赖注入'}。
print(file_events)  # 预期：[('open', False), ('close', True)]。

{'text': '今天练习依赖注入'}
[('open', False), ('close', True)]


## 6 请求失败时，异常经过依赖并继续传播

使用资源的路由或子依赖抛出异常时，已经进入的 yield 依赖也会收到异常。finally 负责清理；如果 except 只是记录异常，记录后应重新 raise，让 FastAPI 继续处理它。

下面增加 observed_note，它依赖 open_note，并记录接收到的 HTTPException 状态码。退出时先结束 observed_note，再结束它使用的 open_note，因此外层依赖退出时仍可使用子依赖提供的资源。

In [11]:
error_events = []


def observed_note(
    note: Annotated[TextIO, Depends(open_note)],
) -> Iterator[TextIO]:
    try:
        yield note
    except HTTPException as error:
        # 此处为观察异常传播而记录；重新抛出以保留原来的接口错误。
        error_events.append((error.status_code, note.closed))
        raise

让一条路由在取得文件后返回“记录不存在”的接口错误，检查客户端状态码和资源清理是否同时符合约定。

In [12]:
@app.get("/missing-note")
def read_missing_note(note: Annotated[TextIO, Depends(observed_note)]):
    raise HTTPException(status_code=404, detail="记录不存在")


file_events.clear()
error_events.clear()
with TestClient(app) as client:
    response = client.get("/missing-note")
# 观察异常时子依赖的文件仍然可用；随后 finally 将文件关闭。
assert response.status_code == 404
assert response.json() == {"detail": "记录不存在"}
assert error_events == [(404, False)]
assert file_events == [("open", False), ("close", True)]
print(response.status_code, response.json())  # 预期：404 {'detail': '记录不存在'}。
print("异常传播：", error_events, "文件状态：", file_events)  # 预期：异常传播： [(404, False)] 文件状态： [('open', False), ('close', True)]。

404

 {'detail': '记录不存在'}
异常传播： [(404, False)] 文件状态： [('open', False), ('close', True)]


## 7 scope 决定 yield 依赖何时退出

使用 yield 的依赖可通过 Depends 的 scope 指定退出时机。这里的范围属于一次请求，不是跨请求缓存的有效期。

| scope 值 | 中文名称／含义 | 退出时机 |
| --- | --- | --- |
| request | 请求范围，yield 依赖的默认值 | 响应发送完成后执行退出代码 |
| function | 路由函数范围 | 路由函数结束后、响应发送前执行退出代码 |

FastAPI 0.121.0 开始支持 scope="function"。默认退出时机曾经调整：0.118.0 恢复到响应发送后；维护旧应用时应核对版本。

下面在路由中先读出普通字符串，后续响应发送不再使用文件，因此可以选择 function。如果响应发送过程还需要资源，应保持资源可用，不能提前关闭。

In [13]:
@app.get("/early-note")
def read_early_note(
    note: Annotated[TextIO, Depends(open_note, scope="function")],
):
    return {"text": note.read()}


file_events.clear()
with TestClient(app) as client:
    response = client.get("/early-note")
# 检查本写法成功读取并关闭文件；TestClient 返回时两种 scope 都已退出。
# 此处的最终列表不用于推断真实网络中的响应发送时间。
assert response.json() == {"text": "今天练习依赖注入"}
assert file_events == [("open", False), ("close", True)]
print(response.json(), file_events)  # 预期：{'text': '今天练习依赖注入'} [('open', False), ('close', True)]。

{'text': '今天练习依赖注入'}

 [('open', False), ('close', True)]


有 yield 子依赖时，还要保证退出代码能访问自己需要的资源：request 范围的依赖不能使用提前退出的 function 范围子依赖；function 范围的依赖可以使用这两种范围的子依赖。

继续复用 open_note，给 observed_note 设置 function 范围；open_note 保持默认 request 范围。异常经过 observed_note 时，底层文件仍然打开。

In [14]:
@app.get("/early-missing-note")
def read_early_missing_note(
    note: Annotated[TextIO, Depends(observed_note, scope="function")],
):
    raise HTTPException(status_code=404, detail="记录不存在")


file_events.clear()
error_events.clear()
with TestClient(app) as client:
    response = client.get("/early-missing-note")
# function 范围的父依赖退出时，request 范围的子依赖仍可使用。
assert response.status_code == 404
assert error_events == [(404, False)]
assert file_events == [("open", False), ("close", True)]
print(response.status_code, error_events, file_events)  # 预期：404 [(404, False)] [('open', False), ('close', True)]。

404

 [(404, False)] [('open', False), ('close', True)]


## 本章小结

（1）Depends 声明由谁提供参数；依赖可以继续声明子依赖，参数校验仍然生效。

（2）默认结果复用发生在一次请求内；use_cache=False 让指定位置重新调用。

（3）需要返回值时声明函数参数，只需执行检查时使用路由或应用的 dependencies。

（4）yield 交付资源，finally 负责清理。仅记录的异常应继续传播；scope 决定退出发生在响应发送之前还是之后。

自查：依赖被两个参数同时使用、新请求再次到来、路由抛出异常时，你能分别说出调用次数和资源的关闭位置吗？

## 练习

（1）把 page_params 的 limit 上限改为 10，检查 /records 和 /tags。验证标准：两个接口传 10 都返回 200，传 11 都返回 422，省略 limit 仍得到 5。

In [ ]:
# 修改本题应用的共享依赖，再分别请求 /records 与 /tags。
# 每条路径比较 limit=10、11 和省略参数三种输入。

（2）为 /cached 添加第三个参数，仍依赖 next_call_number。验证标准：清空 calls 后，一个请求的三个值相同，累计只调用一次；仅给第三个位置添加 use_cache=False 后，本次累计调用两次。

In [ ]:
# 为本题重建应用、计数依赖与三个参数；依次比较默认缓存和第三处禁用缓存。
# 每次请求前清空 calls，观察三个返回值与调用次数。

（3）为 checked_app 增加 /summary，函数返回一个记录总数。验证标准：没有 X-Lab 时返回 400，携带 X-Lab: notebook 时返回 200；路由函数不必增加未使用的检查参数。

In [ ]:
# 在 checked_app 增加 /summary，只返回一个记录总数。
# 对比缺失请求头和 X-Lab: notebook 两次请求。

（4）增加一个依赖 observed_note 的接口，按查询参数 fail 决定返回文本还是抛出 HTTPException(409)。验证标准：成功请求返回原始文本，失败请求返回 409；每个请求结束后都有一次打开和一次关闭，失败时记录 (409, False)。

In [ ]:
# 新增带 fail: bool = False 的路由，注入 observed_note。
# 对比 fail=false 和 fail=true；每次清空并核对事件列表。

## 练习提示与解析

以下对应练习（2），先做两组调用再查看。

提示 1：先确认三处 Depends 指向同一个函数对象。

提示 2：use_cache=False 只改变其所在的第三个位置；观察默认位置的已缓存值。

### 练习（2）参考解析

每次请求前清空 calls，三个位置都使用默认缓存时，返回值依次为 1、1、1，依赖调用一次。仅第三处设置 use_cache=False 时，前两处仍共用 1，第三处重新执行得到 2，总调用次数为 2。新请求会建立自己的依赖缓存；如果不清空用于累计观察的 calls，数值会继续增长，不能把列表本身当成框架的请求内缓存。

## 参考与引用来源

- **FastAPI 官方文档（fastapi.tiangolo.com）**：
  - [Dependencies](https://fastapi.tiangolo.com/tutorial/dependencies/)，定位 Create a dependency、Import Depends 与 Declare the dependency，支持参数注入、复用与校验；[Sub-dependencies](https://fastapi.tiangolo.com/tutorial/dependencies/sub-dependencies/#using-the-same-dependency-multiple-times)，定位 Using the same dependency multiple times，支持子依赖、请求内缓存与 use_cache=False。
  - [Dependencies in path operation decorators](https://fastapi.tiangolo.com/tutorial/dependencies/dependencies-in-path-operation-decorators/)，定位 Add dependencies to the path operation decorator、Dependencies errors and return values；[Global Dependencies](https://fastapi.tiangolo.com/tutorial/dependencies/global-dependencies/)，支持路由与应用级依赖。
  - [Dependencies with yield](https://fastapi.tiangolo.com/tutorial/dependencies/dependencies-with-yield/)，定位 A dependency with yield and try、Always raise in Dependencies with yield and except、Early exit and scope 及 scope for sub-dependencies，支持资源退出、异常传播及作用域约束；[Advanced Dependencies](https://fastapi.tiangolo.com/advanced/advanced-dependencies/#dependencies-with-yield-and-scope)，定位 Dependencies with yield and scope 及紧随的 StreamingResponse 技术说明，支持 0.121.0、0.118.0 版本边界。
  - [Depends API](https://fastapi.tiangolo.com/reference/dependencies/#fastapi.Depends)，定位 use_cache 与 scope 参数；[Header Parameters](https://fastapi.tiangolo.com/tutorial/header-params/#automatic-conversion)，定位 Automatic conversion；[Testing](https://fastapi.tiangolo.com/tutorial/testing/#using-testclient)，支持本章客户端调用与断言写法；[Numeric Validations](https://fastapi.tiangolo.com/tutorial/path-params-numeric-validations/#number-validations-greater-than-or-equal)，支持 ge、le 数值约束。
- **Python 3.12 官方文档（docs.python.org）**：[tempfile.TemporaryFile](https://docs.python.org/3.12/library/tempfile.html#tempfile.TemporaryFile)，支持临时文件、文本模式与关闭后删除；[Defining Clean-up Actions](https://docs.python.org/3.12/tutorial/errors.html#defining-clean-up-actions)，支持 finally 清理和异常继续传播；[typing](https://docs.python.org/3.12/library/typing.html#annotating-generators-and-coroutines)，定位 Annotating generators and coroutines 及 ABCs for working with IO，支持 Iterator 与 TextIO 标注。
- **Starlette 官方文档（starlette.dev）**：[TestClient](https://starlette.dev/testclient/)，定位 TestClient 的 with 上下文用法，支持本章客户端的使用范围。